In [39]:
import os 

In [9]:
%pwd 
os.chdir("../")

In [40]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    train_data_path: Path
    test_data_path: Path

In [41]:
from src.student_Performance_p1.constants import *
from src.student_Performance_p1.utils.common import read_yaml, create_directories


In [43]:
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH, schema_filepath=SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        # Access data_transformation section from config.yaml
        config = self.config.data_transformation

        # Create data transformation folder
        create_directories([config.root_dir])

        # Create data transformation config object
        data_transformation_config = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            train_data_path=config.transformed_train_data,
            test_data_path=config.transformed_test_data
        )

        return data_transformation_config
    
    

In [44]:
import os 
from src.student_Performance_p1 import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [45]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
    
    def initiate_data_transformation(self):
        logger.info("Reading the data from the source")
        df = pd.read_csv(self.config.data_path)

        logger.info("Splitting the data into train and test sets")
        train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

        logger.info("Saving the train and test sets to the specified paths")
        train_set.to_csv(self.config.train_data_path, index=False)
        test_set.to_csv(self.config.test_data_path, index=False)

        logger.info(f"Train data saved at: {self.config.train_data_path}")
        logger.info(f"Test data saved at: {self.config.test_data_path}")

In [46]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.initiate_data_transformation()
except Exception as e:
    logger.exception(f"Error in data transformation process: {e}")
    
   

[2026-05-26 21:51:21,214: INFO: common: YAML file 'config\config.yaml' Loading successfully.]
[2026-05-26 21:51:21,217: INFO: common: YAML file 'params.yaml' Loading successfully.]
[2026-05-26 21:51:21,220: INFO: common: YAML file 'schema.yaml' Loading successfully.]
[2026-05-26 21:51:21,222: INFO: common: Directories created successfully: ['artifacts']]
[2026-05-26 21:51:21,226: INFO: common: Directories created successfully: ['artifacts/data_transformation']]
[2026-05-26 21:51:21,228: INFO: 4043773373: Reading the data from the source]
[2026-05-26 21:51:21,271: INFO: 4043773373: Splitting the data into train and test sets]
[2026-05-26 21:51:21,286: INFO: 4043773373: Saving the train and test sets to the specified paths]
[2026-05-26 21:51:21,527: INFO: 4043773373: Train data saved at: artifacts/data_transformation/transformed_train_data.csv]
[2026-05-26 21:51:21,530: INFO: 4043773373: Test data saved at: artifacts/data_transformation/transformed_test_data.csv]
